In [3]:
%pip install langchain-ollama langchain-community langchain-huggingface faiss-cpu mysql-connector-python pandas


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import mysql.connector
import random

# Database configuration
db_config = {
    'host': 'localhost',
    'user': 'root',     # Your MySQL username
    'password': 'root', # Your MySQL password
    'database': 'company'
}

def populate_mysql():
    conn = mysql.connector.connect(**db_config)
    cursor = conn.cursor()
    
    depts = ['Engineering', 'Marketing', 'Sales', 'HR', 'Finance', 'Product']
    skills_pool = ['Python', 'Java', 'React', 'SQL', 'AWS', 'SEO', 'Tableau', 'Docker', 'Kubernetes', 'Project Management']
    
    # Generate 100 instances
    employee_data = []
    for i in range(1, 101):
        name = f"Employee_{i}"
        dept = random.choice(depts)
        skills = ", ".join(random.sample(skills_pool, 3))
        bio = f"Experienced professional in {dept} with a focus on {skills.split(',')[0]}."
        employee_data.append((name, dept, skills, bio))
    
    query = "INSERT INTO employees (name, department, skills, bio) VALUES (%s, %s, %s, %s)"
    cursor.executemany(query, employee_data)
    conn.commit()
    cursor.close()
    conn.close()
    print("100 employee records inserted into MySQL.")

populate_mysql()

100 employee records inserted into MySQL.


In [6]:
import mysql.connector

%pip install --upgrade --force-reinstall pyarrow

import pandas as pd
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.schema import Document

  Using cached pyarrow-25.0.1-cp314-cp314-win_amd64.whl.metadata (3.0 kB)
Using cached pyarrow-25.0.1-cp314-cp314-win_amd64.whl (28.6 MB)
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 25.0.1
    Uninstalling pyarrow-25.0.1:
      Successfully uninstalled pyarrow-25.0.1
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
C:\Users\Asus\AppData\Local\Temp\ipykernel_33128\4137090350.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [8]:
# 1. Database Connection (Replace with your credentials)
db_config = {
    'host': 'localhost',
    'user': 'root',
    'password': 'root',
    'database': 'company'
}

conn = mysql.connector.connect(**db_config)
df = pd.read_sql_query("SELECT * FROM employees", conn)
conn.close()

C:\Users\Asus\AppData\Local\Temp\ipykernel_33128\2169366687.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query("SELECT * FROM employees", conn)


In [9]:
print(type(df))

<class 'pandas.DataFrame'>


In [10]:
# 2. Convert database rows into RAG Documents
documents = [
    Document(
        page_content=f"Employee: {row['name']}. Dept: {row['department']}. Skills: {row['skills']}. Background: {row['bio']}",
        metadata={"id": row['id']}
    ) for _, row in df.iterrows()
]

In [11]:
print(documents[0])

page_content='Employee: Employee_1. Dept: Engineering. Skills: AWS, SQL, Java. Background: Experienced professional in Engineering with a focus on AWS.' metadata={'id': 1}


In [12]:
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

c:\Users\Asus\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2392.72it/s]


In [14]:
# 4. Create Vector Store
vector_db = FAISS.from_documents(documents, embeddings)
print(f"FAISS index built with {len(documents)} employee records.")

FAISS index built with 100 employee records.


In [15]:
%pip install langchain langchain-community langchain-ollama langchain-huggingface faiss-cpu mysql-connector-python pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
%pip install --upgrade langchain langchain-community langchain-ollama

  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.18
    Uninstalling langchain-1.3.18:
      Successfully uninstalled langchain-1.3.18
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
